In [2]:
import pandas as pd
import re
import pickle
import os
import unicodedata
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
data = pd.read_csv("../data/np20ng.csv")

In [4]:
data.head()

,source,category,heading,content
0,Gorkhapatra,Education,"अटेर गर्दै कतिपय विद्यालय, चैतमै भर्ना अभियान ...","काठमाडौं,चैत्र ७ गते । उपत्याकाका कतिपय विद्या..."
1,Gorkhapatra,Education,"निर्वाचन प्रचारमा विद्यालय, शिक्षक र विद्यार्...","प्रकाश सिलवाल काठमाडौँ, चैत ६ गते । विगतका निर..."
2,Gorkhapatra,Education,विद्यार्थीलाई परम्परागत सीप सिकाउँदै विद्यालय,"अमरराज नहर्की तनहुँ, चैत्र ५ गते । तनहुँको एक ..."
3,Gorkhapatra,Education,आन्दोलन नगर्न शिक्षामन्त्रीको आग्रह,"गोरखापत्र समाचारदाता काठमाडौँ, चैत ३ गते । शिक..."
4,Gorkhapatra,Education,विश्वविद्यालयमा हडताल नगर्न शिक्षामन्त्रीको आग्रह,"काठमाडौं, चैत २ गते। शिक्षा विज्ञान तथा प्रविध..."


In [6]:
def validate_dataframe(df):

    # remove rows with empty or NaN content
    initial = len(df)
    df = df.dropna(subset=["content"])
    df = df[
        df["content"].str.strip().ne("")
    ]  # remove rows with empty content after stripping whitespace
    df = df.drop_duplicates(
        subset=["heading", "content"]
    )  # remove duplicate rows based on heading and content
    print(
        f"Validation: {initial} → {len(df)} rows ({initial - len(df)} removed)"
    )  # print number of rows removed
    return df


data = validate_dataframe(data)

Validation: 231216 → 231128 rows (88 removed)


In [7]:
def clean_and_format(entry: dict) -> str:

    # combine header and body
    header = f"Category: {entry.get('category', 'General')} | Title: {entry.get('heading', '')}"
    body = entry.get("content", "")
    full_text = f"{header}\n{body}"

    # unicode normalization
    full_text = unicodedata.normalize("NFKC", full_text)

    # fix known encoding artifact
    full_text = full_text.replace("¥", "र्")

    # collapse whitespace
    full_text = re.sub(r"\s+", " ", full_text).strip()

    return full_text

In [8]:
# process and chunk
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ।", "।", "|", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
)

In [9]:
all_chunks = []
for row in data.itertuples(index=False):
    entry = row._asdict()
    cleaned_text = clean_and_format(entry)

    metadata = {
        "source": entry.get("source", "Unknown"),  # outlet name
        "category": entry.get("category", "General"),  # news category
        "heading": entry.get("heading", ""),  # article title
    }
    
    # create chunks
    doc_chunks = text_splitter.create_documents([cleaned_text], metadatas=[metadata])
    all_chunks.extend(doc_chunks)

print(f"Total Source Docs: {len(data)}")
print(f"Total Chunks Generated: {len(all_chunks)}")
print(f"Average Chunks per Doc: {len(all_chunks)/len(data):.2f}")

Total Source Docs: 231128
Total Chunks Generated: 593779
Average Chunks per Doc: 2.57


In [10]:
# save chunks
save_path = "../data/chunks/nepali_news_chunks.pkl"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "wb") as f:
    pickle.dump(all_chunks, f)